# 10-Fold Heterogeneous GraphSAGE Training for CXR KG

This notebook trains one heterogeneous GraphSAGE model per fold using the fold-specific KG files generated by `radgraph_biomedclip_graphrag_kg.ipynb`.

The prediction task is dense image-label link prediction. Each image node is scored against the 14 shared label nodes (`normal` + 13 CheXpert findings), producing the same output space as the BiomedCLIP + MLP baseline.

Important leakage control: by default, `has_finding` and `rev_has_finding` edges are **excluded from message passing** and are used only as supervision labels (`data["image"].y`). This avoids the trivial training-time shortcut where an image receives its own ground-truth label edge, while a new query image has no such edge.


In [ ]:
# Optional dependency installation. Set True in a fresh environment.
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    import sys
    packages = [
        "torch", "torch_geometric", "faiss-cpu", "pandas", "numpy", "scikit-learn", "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])


In [ ]:
import copy
import json
import math
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.nn import HeteroConv, SAGEConv
except ImportError as exc:
    raise ImportError("Install torch_geometric before running this notebook.") from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # PyG is most reliable on CUDA/CPU; avoid MPS fallback

DATA_ROOT = Path("/data/liangz2/openi/multi_kg") if Path("/data/liangz2/openi/multi_kg").exists() else Path.cwd()
KG_10FOLD_DIR = DATA_ROOT / "kg_10fold"
OUTPUT_ROOT = DATA_ROOT / "kg_graphsage_10fold"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FOLD_INDICES = list(range(10))
SKIP_COMPLETED = True
FORCE_RERUN_FOLDS = []

# Training config.
VAL_FRAC = 0.10
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.20
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 100
PATIENCE = 12
GRAD_CLIP_NORM = 5.0

# Decoding config.
DECODER = "dot"  # "dot" or "mlp"

# Retrieval/query config.
QUERY_TOP_K = 50
TEST_QUERY_BATCH_SIZE = 512

# Leakage control. Keep False for the primary experiment.
INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING = True

SCORING_LABELS = (
    "normal",
    "enlarged cardiomediastinum", "cardiomegaly", "lung opacity", "lung lesion", "edema",
    "consolidation", "pneumonia", "atelectasis", "pneumothorax",
    "pleural effusion", "pleural other", "fracture", "support devices",
)
TARGET_COLUMNS = ["y_" + label.replace(" ", "_") for label in SCORING_LABELS]
FINDING_LABELS = SCORING_LABELS[1:]

print("DEVICE", DEVICE)
print("KG_10FOLD_DIR", KG_10FOLD_DIR)
print("OUTPUT_ROOT", OUTPUT_ROOT)


## Metrics and Utility Functions

In [ ]:
def sigmoid_np(logits: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-logits))


def evaluate_prob_matrix(probs: np.ndarray, targets: np.ndarray, thresholds: Optional[np.ndarray] = None) -> pd.DataFrame:
    y = targets.astype(int)
    if thresholds is None:
        thresholds = np.full((probs.shape[1],), 0.5, dtype=np.float32)
    rows = []
    for j, label in enumerate(SCORING_LABELS):
        if len(np.unique(y[:, j])) < 2:
            continue
        pred = (probs[:, j] >= thresholds[j]).astype(int)
        rows.append({
            "label": label,
            "auroc": roc_auc_score(y[:, j], probs[:, j]),
            "auprc": average_precision_score(y[:, j], probs[:, j]),
            "f1": f1_score(y[:, j], pred, zero_division=0),
            "threshold": float(thresholds[j]),
            "support": int(y[:, j].sum()),
        })
    return pd.DataFrame(rows)


def macro_metric(metrics: pd.DataFrame, metric: str = "auroc", disease_only: bool = True) -> float:
    if metrics.empty:
        return float("nan")
    frame = metrics[metrics["label"] != "normal"] if disease_only else metrics
    if frame.empty:
        return float("nan")
    return float(frame[metric].mean())


def tune_f1_thresholds(probs: np.ndarray, targets: np.ndarray, grid: Optional[np.ndarray] = None) -> np.ndarray:
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)
    thresholds = np.full((probs.shape[1],), 0.5, dtype=np.float32)
    y = targets.astype(int)
    for j in range(probs.shape[1]):
        if len(np.unique(y[:, j])) < 2:
            continue
        best_f1, best_thr = -1.0, 0.5
        for thr in grid:
            pred = (probs[:, j] >= thr).astype(int)
            value = f1_score(y[:, j], pred, zero_division=0)
            if value > best_f1:
                best_f1, best_thr = value, float(thr)
        thresholds[j] = best_thr
    return thresholds


def metric_summary(metrics: pd.DataFrame, fold: int, method: str, extra: Optional[Dict] = None) -> Dict:
    disease = metrics[metrics["label"] != "normal"] if not metrics.empty else metrics
    out = {
        "fold": int(fold),
        "method": method,
        "disease_macro_auroc": float(disease["auroc"].mean()) if len(disease) else float("nan"),
        "disease_macro_auprc": float(disease["auprc"].mean()) if len(disease) else float("nan"),
        "disease_macro_f1": float(disease["f1"].mean()) if len(disease) else float("nan"),
        "all_macro_auroc": float(metrics["auroc"].mean()) if len(metrics) else float("nan"),
        "all_macro_auprc": float(metrics["auprc"].mean()) if len(metrics) else float("nan"),
        "all_macro_f1": float(metrics["f1"].mean()) if len(metrics) else float("nan"),
        "num_scored_labels": int(len(metrics)),
    }
    if extra:
        out.update(extra)
    return out


def save_json(obj: Dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")


## Load Fold Graphs

The fold graph is expected at `kg_10fold/fold{i}/kg_heterodata_fold{i}.pt`. If the KG builder saved a dictionary fallback instead of a PyG `HeteroData`, this notebook converts it back into `HeteroData` for training.


In [ ]:
def torch_load_any(path: Path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def dict_to_heterodata(obj: Dict) -> HeteroData:
    data = HeteroData()
    for node_type, x in obj.get("x_dict", {}).items():
        data[node_type].x = x.float()
        node_ids = obj.get("node_ids", {}).get(node_type)
        if node_ids is not None:
            data[node_type].node_id = list(node_ids)
        if node_type in obj.get("y_dict", {}):
            data[node_type].y = obj["y_dict"][node_type].float()
    for etype, edge_index in obj.get("edge_index_dict", {}).items():
        data[etype].edge_index = edge_index.long()
        edge_weight = obj.get("edge_weight_dict", {}).get(etype)
        if edge_weight is not None:
            data[etype].edge_weight = edge_weight.float()
    return data


def load_fold_data(fold: int) -> Tuple[HeteroData, pd.DataFrame, pd.DataFrame, np.ndarray]:
    fold_dir = KG_10FOLD_DIR / f"fold{fold}"
    kg_path = fold_dir / f"kg_heterodata_fold{fold}.pt"
    train_meta_path = fold_dir / f"train_metadata_fold{fold}.csv"
    test_meta_path = fold_dir / f"test_query_metadata_fold{fold}.csv"
    test_emb_path = fold_dir / f"test_query_embeddings_fold{fold}.npy"
    if not kg_path.exists():
        raise FileNotFoundError(f"Missing fold KG file: {kg_path}")
    obj = torch_load_any(kg_path, map_location="cpu")
    data = obj if isinstance(obj, HeteroData) else dict_to_heterodata(obj)
    train_df = pd.read_csv(train_meta_path)
    test_df = pd.read_csv(test_meta_path)
    test_embeddings = np.load(test_emb_path).astype("float32")
    return data, train_df, test_df, test_embeddings


def label_names_from_data(data: HeteroData) -> List[str]:
    if hasattr(data["label"], "label_name"):
        return [str(x) for x in data["label"].label_name]
    if hasattr(data["label"], "node_id"):
        names = []
        for node_id in data["label"].node_id:
            text = str(node_id)
            names.append(text.split("label::", 1)[-1] if text.startswith("label::") else text)
        return names
    raise RuntimeError("Label node names are missing from HeteroData. Regenerate KG with label_name/node_id metadata.")


def label_indices_for_scoring(data: HeteroData) -> List[int]:
    label_names = label_names_from_data(data)
    lookup = {name: i for i, name in enumerate(label_names)}
    missing = [label for label in SCORING_LABELS if label not in lookup]
    if missing:
        raise RuntimeError(f"Missing label nodes: {missing}. Available labels: {label_names}")
    return [lookup[label] for label in SCORING_LABELS]


def split_train_val_indices(num_images: int, val_frac: float, seed: int) -> Tuple[torch.Tensor, torch.Tensor]:
    rng = np.random.default_rng(seed)
    indices = np.arange(num_images)
    rng.shuffle(indices)
    val_size = max(1, int(round(num_images * val_frac)))
    val_idx = np.sort(indices[:val_size])
    train_idx = np.sort(indices[val_size:])
    return torch.from_numpy(train_idx).long(), torch.from_numpy(val_idx).long()


## Heterogeneous GraphSAGE Model

In [ ]:
class HeteroGraphSAGE(nn.Module):
    def __init__(
        self,
        metadata,
        input_dims: Dict[str, int],
        hidden_dim: int = HIDDEN_DIM,
        num_layers: int = NUM_LAYERS,
        dropout: float = DROPOUT,
        decoder: str = DECODER,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.decoder = decoder
        node_types, edge_types = metadata
        self.proj = nn.ModuleDict({
            node_type: nn.Linear(int(input_dims[node_type]), hidden_dim)
            for node_type in node_types
            if node_type in input_dims
        })
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv((hidden_dim, hidden_dim), hidden_dim)
                for edge_type in edge_types
            }, aggr="sum")
            self.convs.append(conv)
            self.norms.append(nn.ModuleDict({node_type: nn.LayerNorm(hidden_dim) for node_type in node_types if node_type in input_dims}))
        if decoder == "mlp":
            self.decoder_mlp = nn.Sequential(
                nn.Linear(2 * hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1)
            )
        elif decoder != "dot":
            raise ValueError(f"Unknown decoder: {decoder}")

    def forward(self, x_dict, edge_index_dict):
        z = {}
        for node_type, x in x_dict.items():
            if node_type not in self.proj:
                continue
            z[node_type] = F.relu(self.proj[node_type](x.float()))
        for conv, norm_dict in zip(self.convs, self.norms):
            out = conv(z, edge_index_dict)
            next_z = {}
            for node_type, x in z.items():
                y = out.get(node_type, x)
                y = F.relu(y)
                y = norm_dict[node_type](y)
                y = F.dropout(y, p=self.dropout, training=self.training)
                next_z[node_type] = x + y if x.shape == y.shape else y
            z = next_z
        return z

    def decode_image_label_logits(self, z_dict, label_indices: Sequence[int]) -> torch.Tensor:
        image_z = z_dict["image"]
        label_z = z_dict["label"][torch.as_tensor(label_indices, device=image_z.device)]
        if self.decoder == "dot":
            return image_z @ label_z.t()
        image_expanded = image_z[:, None, :].expand(-1, len(label_indices), -1)
        label_expanded = label_z[None, :, :].expand(image_z.size(0), -1, -1)
        pair = torch.cat([image_expanded, label_expanded], dim=-1)
        return self.decoder_mlp(pair).squeeze(-1)


def message_passing_edge_index_dict(data: HeteroData, include_has_finding: bool = INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING):
    edge_dict = {}
    for edge_type in data.edge_types:
        relation = edge_type[1]
        if not include_has_finding and (relation == "has_finding" or relation == "rev_has_finding"):
            continue
        if hasattr(data[edge_type], "edge_index"):
            edge_dict[edge_type] = data[edge_type].edge_index
    return edge_dict


def input_dims_from_data(data: HeteroData) -> Dict[str, int]:
    return {node_type: int(data[node_type].x.size(-1)) for node_type in data.node_types if hasattr(data[node_type], "x")}


def pos_weight_from_targets(y: torch.Tensor) -> torch.Tensor:
    pos = y.sum(dim=0)
    neg = y.size(0) - pos
    return torch.clamp(neg / torch.clamp(pos, min=1.0), min=1.0, max=50.0)


## Query Graph Construction

At test time, the query images are appended as new image nodes. Their `has_finding` edges are never added. The only query graph edges are FAISS-style `similar_to` and `rev_similar_to` edges to training images.


In [ ]:
def l2_normalize_np(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return (x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)).astype("float32")


def topk_inner_product(train_embeddings: np.ndarray, query_embeddings: np.ndarray, top_k: int, batch_size: int = TEST_QUERY_BATCH_SIZE):
    train = l2_normalize_np(train_embeddings.astype("float32"))
    query = l2_normalize_np(query_embeddings.astype("float32"))
    all_sims, all_idx = [], []
    try:
        import faiss
        index = faiss.IndexFlatIP(train.shape[1])
        index.add(np.ascontiguousarray(train))
        for start in tqdm(range(0, len(query), batch_size), desc="query FAISS"):
            sims, idx = index.search(np.ascontiguousarray(query[start:start + batch_size]), min(top_k, len(train)))
            all_sims.append(sims.astype("float32"))
            all_idx.append(idx.astype("int64"))
    except ImportError:
        for start in tqdm(range(0, len(query), batch_size), desc="query dense top-k"):
            scores = query[start:start + batch_size] @ train.T
            idx = np.argpartition(-scores, kth=min(top_k, scores.shape[1] - 1), axis=1)[:, :top_k]
            sorted_order = np.take_along_axis(scores, idx, axis=1).argsort(axis=1)[:, ::-1]
            idx = np.take_along_axis(idx, sorted_order, axis=1)
            sims = np.take_along_axis(scores, idx, axis=1)
            all_sims.append(sims.astype("float32"))
            all_idx.append(idx.astype("int64"))
    return np.concatenate(all_sims, axis=0), np.concatenate(all_idx, axis=0)


def append_edge_index(data: HeteroData, edge_type, edge_index_new: torch.Tensor, edge_weight_new: Optional[torch.Tensor] = None):
    store = data[edge_type]
    if hasattr(store, "edge_index"):
        store.edge_index = torch.cat([store.edge_index.cpu(), edge_index_new.cpu()], dim=1)
    else:
        store.edge_index = edge_index_new.cpu()
    if edge_weight_new is not None:
        if hasattr(store, "edge_weight"):
            store.edge_weight = torch.cat([store.edge_weight.cpu(), edge_weight_new.cpu()], dim=0)
        else:
            store.edge_weight = edge_weight_new.cpu()


def target_matrix_from_metadata(frame: pd.DataFrame) -> np.ndarray:
    missing = [col for col in TARGET_COLUMNS if col not in frame.columns]
    if missing:
        raise RuntimeError(f"Missing target columns in test metadata: {missing}")
    return frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)


def append_query_images_to_graph(data: HeteroData, test_df: pd.DataFrame, test_embeddings: np.ndarray, top_k: int = QUERY_TOP_K) -> Tuple[HeteroData, int, np.ndarray]:
    query_data = copy.deepcopy(data).cpu()
    train_count = int(query_data["image"].x.size(0))
    train_embeddings = query_data["image"].x.cpu().numpy().astype("float32")
    test_targets = target_matrix_from_metadata(test_df)

    query_data["image"].x = torch.cat([query_data["image"].x.float(), torch.from_numpy(test_embeddings).float()], dim=0)
    if hasattr(query_data["image"], "y"):
        query_data["image"].y = torch.cat([query_data["image"].y.float(), torch.from_numpy(test_targets).float()], dim=0)
    else:
        query_data["image"].y = torch.from_numpy(test_targets).float()

    sims, idx = topk_inner_product(train_embeddings, test_embeddings, top_k=top_k)
    src, dst, weights = [], [], []
    for qi in range(idx.shape[0]):
        query_node = train_count + qi
        for rank in range(idx.shape[1]):
            train_node = int(idx[qi, rank])
            src.append(query_node)
            dst.append(train_node)
            weights.append(float(sims[qi, rank]))
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_weight = torch.tensor(weights, dtype=torch.float32)
    append_edge_index(query_data, ("image", "similar_to", "image"), edge_index, edge_weight)
    append_edge_index(query_data, ("image", "rev_similar_to", "image"), edge_index.flip(0), edge_weight)
    return query_data, train_count, test_targets


## Train and Evaluate One Fold

In [ ]:
def fold_dirs(fold: int) -> Dict[str, Path]:
    root = OUTPUT_ROOT / f"fold{fold}"
    dirs = {
        "root": root,
        "models": root / "models",
        "metrics": root / "metrics",
        "predictions": root / "predictions",
        "logs": root / "logs",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def fold_complete(fold: int, dirs: Dict[str, Path]) -> bool:
    return (dirs["root"] / "FOLD_COMPLETE.json").exists() and (dirs["metrics"] / "graphsage_test_metrics.csv").exists()


def save_predictions(path: Path, test_df: pd.DataFrame, probs: np.ndarray, targets: np.ndarray) -> pd.DataFrame:
    rows = []
    for i, row in test_df.reset_index(drop=True).iterrows():
        out = {
            "study_id": str(row.get("study_id", "")),
            "subject_id": str(row.get("subject_id", "")),
            "image_path": str(row.get("image_path", "")),
        }
        for j, label in enumerate(SCORING_LABELS):
            out[f"prob_{label}"] = float(probs[i, j])
            out[f"target_{label}"] = int(targets[i, j] >= 0.5)
        rows.append(out)
    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(path, index=False)
    return pred_df


def train_one_fold(fold: int) -> List[Dict]:
    dirs = fold_dirs(fold)
    if SKIP_COMPLETED and fold not in FORCE_RERUN_FOLDS and fold_complete(fold, dirs):
        print(f"Fold {fold}: already complete, skipping.")
        summary_csv = dirs["metrics"] / "graphsage_method_summary.csv"
        if summary_csv.exists():
            return pd.read_csv(summary_csv).to_dict(orient="records")
        summary_path = dirs["metrics"] / "graphsage_summary.json"
        return json.loads(summary_path.read_text()) if summary_path.exists() else [{"fold": fold, "skipped": True}]

    data, train_df, test_df, test_embeddings = load_fold_data(fold)
    label_indices = label_indices_for_scoring(data)
    if not hasattr(data["image"], "y"):
        raise RuntimeError("data['image'].y is missing. Regenerate fold KG with train labels.")

    num_images = int(data["image"].x.size(0))
    train_idx, val_idx = split_train_val_indices(num_images, VAL_FRAC, seed=SEED + fold)
    y = data["image"].y.float()
    train_y = y[train_idx]
    val_y = y[val_idx]
    pos_weight = pos_weight_from_targets(train_y).to(DEVICE)

    mp_edges = message_passing_edge_index_dict(data, include_has_finding=INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING)
    data = data.to(DEVICE)
    mp_edges = {etype: edge_index.to(DEVICE) for etype, edge_index in mp_edges.items()}

    model = HeteroGraphSAGE(
        metadata=(data.node_types, list(mp_edges.keys())),
        input_dims=input_dims_from_data(data),
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        decoder=DECODER,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_score = -float("inf")
    best_state = None
    best_epoch = -1
    stale_epochs = 0
    logs = []

    train_idx_dev = train_idx.to(DEVICE)
    val_idx_dev = val_idx.to(DEVICE)
    label_indices_dev = label_indices

    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        z = model(data.x_dict, mp_edges)
        logits = model.decode_image_label_logits(z, label_indices_dev)
        loss = criterion(logits[train_idx_dev], data["image"].y[train_idx_dev].float())
        loss.backward()
        if GRAD_CLIP_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            z = model(data.x_dict, mp_edges)
            logits = model.decode_image_label_logits(z, label_indices_dev)
            val_logits = logits[val_idx_dev].detach().cpu().numpy().astype("float32")
            val_probs = sigmoid_np(val_logits)
            val_metrics = evaluate_prob_matrix(val_probs, val_y.numpy())
            val_macro_auroc = macro_metric(val_metrics, "auroc", disease_only=True)
            train_logits = logits[train_idx_dev].detach().cpu().numpy().astype("float32")
            train_probs = sigmoid_np(train_logits)
            train_metrics = evaluate_prob_matrix(train_probs, train_y.numpy())
            train_macro_auroc = macro_metric(train_metrics, "auroc", disease_only=True)

        log_row = {
            "fold": int(fold),
            "epoch": int(epoch),
            "loss": float(loss.detach().cpu()),
            "train_disease_macro_auroc": float(train_macro_auroc),
            "val_disease_macro_auroc": float(val_macro_auroc),
        }
        logs.append(log_row)
        print(f"fold={fold} epoch={epoch} loss={log_row['loss']:.4f} val_disease_macro_auroc={val_macro_auroc:.4f}")

        if np.isfinite(val_macro_auroc) and val_macro_auroc > best_score:
            best_score = float(val_macro_auroc)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = int(epoch)
            stale_epochs = 0
        else:
            stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Fold {fold}: early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    pd.DataFrame(logs).to_csv(dirs["logs"] / "graphsage_training_log.csv", index=False)

    # Tune thresholds on validation nodes.
    model.eval()
    with torch.no_grad():
        z = model(data.x_dict, mp_edges)
        all_logits = model.decode_image_label_logits(z, label_indices_dev)
        val_probs = sigmoid_np(all_logits[val_idx_dev].detach().cpu().numpy().astype("float32"))
    thresholds = tune_f1_thresholds(val_probs, val_y.numpy())
    pd.DataFrame({"label": SCORING_LABELS, "threshold": thresholds}).to_csv(dirs["metrics"] / "graphsage_val_f1_thresholds.csv", index=False)

    # Test-time query graph.
    data_cpu, _, test_df, test_embeddings = load_fold_data(fold)
    query_data, train_count, test_targets = append_query_images_to_graph(data_cpu, test_df, test_embeddings, top_k=QUERY_TOP_K)
    query_edges = message_passing_edge_index_dict(query_data, include_has_finding=INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING)
    query_data = query_data.to(DEVICE)
    query_edges = {etype: edge_index.to(DEVICE) for etype, edge_index in query_edges.items()}
    with torch.no_grad():
        z = model(query_data.x_dict, query_edges)
        query_logits_all = model.decode_image_label_logits(z, label_indices_dev)
        test_logits = query_logits_all[train_count:].detach().cpu().numpy().astype("float32")
        test_probs = sigmoid_np(test_logits)

    metrics_fixed = evaluate_prob_matrix(test_probs, test_targets)
    metrics_tuned = evaluate_prob_matrix(test_probs, test_targets, thresholds=thresholds)
    metrics_fixed.to_csv(dirs["metrics"] / "graphsage_test_metrics.csv", index=False)
    metrics_tuned.to_csv(dirs["metrics"] / "graphsage_test_metrics_val_tuned_thresholds.csv", index=False)
    save_predictions(dirs["predictions"] / "graphsage_test_predictions.csv", test_df, test_probs, test_targets)

    model_path = dirs["models"] / f"hetero_graphsage_fold{fold}.pt"
    torch.save({
        "state_dict": model.state_dict(),
        "fold": int(fold),
        "labels": list(SCORING_LABELS),
        "label_indices": list(label_indices),
        "input_dims": input_dims_from_data(data),
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "decoder": DECODER,
        "include_has_finding_in_message_passing": INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING,
        "query_top_k": QUERY_TOP_K,
        "best_epoch": int(best_epoch),
        "best_val_disease_macro_auroc": float(best_score),
    }, model_path)

    summary = metric_summary(metrics_fixed, fold, "hetero_graphsage", extra={
        "threshold_mode": "fixed_0.5",
        "model_path": str(model_path),
        "metrics_path": str(dirs["metrics"] / "graphsage_test_metrics.csv"),
        "best_epoch": int(best_epoch),
        "best_val_disease_macro_auroc": float(best_score),
        "query_top_k": int(QUERY_TOP_K),
        "include_has_finding_in_message_passing": bool(INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING),
    })
    summary_tuned = metric_summary(metrics_tuned, fold, "hetero_graphsage", extra={
        "threshold_mode": "val_tuned_f1",
        "model_path": str(model_path),
        "metrics_path": str(dirs["metrics"] / "graphsage_test_metrics_val_tuned_thresholds.csv"),
        "best_epoch": int(best_epoch),
        "best_val_disease_macro_auroc": float(best_score),
        "query_top_k": int(QUERY_TOP_K),
        "include_has_finding_in_message_passing": bool(INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING),
    })
    pd.DataFrame([summary, summary_tuned]).to_csv(dirs["metrics"] / "graphsage_method_summary.csv", index=False)
    save_json({"fixed_0.5": summary, "val_tuned_f1": summary_tuned}, dirs["metrics"] / "graphsage_summary.json")
    save_json({"fold": int(fold), "status": "complete", "model_path": str(model_path)}, dirs["root"] / "FOLD_COMPLETE.json")
    return [summary, summary_tuned]


## Run 10-Fold Training

In [ ]:
all_summaries = []
for fold in FOLD_INDICES:
    print("\n===== Fold", fold, "=====")
    summary = train_one_fold(fold)
    all_summaries.append(summary)
    pd.DataFrame(all_summaries).to_csv(OUTPUT_ROOT / "graphsage_10fold_summary_progress.csv", index=False)

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(OUTPUT_ROOT / "graphsage_10fold_summary.csv", index=False)
display(summary_df)


## Notes for Comparison

Compare `graphsage_10fold_summary.csv` against your existing BiomedCLIP/FAISS/RoentGen 10-fold summaries. The most direct comparison metric is disease macro AUROC.

If you want to test the effect of neighbor label topology, you can set:

```python
INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING = True
```

but treat that as an ablation with caution because it can leak labels into train image representations. The primary setting should keep it `False` and use `has_finding` only as supervision.
